# 3. Baselines: floors, image VAE, supervised references

This notebook produces everything the URL VAE is compared against:

| Model | Trained on | Role |
| --- | --- | --- |
| Floor A: URL (`url_len`), image (`modules`, `dark_frac`) | benign train | headline floor each VAE must beat |
| Floor B: URL adds `has_scheme`, `host_is_ip`, `n_host_labels`, `path_len`, `digit_ratio` | benign train | harder floor, reported alongside |
| Image VAE (65 x 65 module grid) | benign train | comparison model on the QR image |
| Supervised image CNN, supervised URL character CNN | benign + malicious train | references (ceilings) for the gap-closed ratio |

Every model writes `results/scores/<run>.parquet` (item_id + score, higher = more malicious, validation and test rows).
The URL VAE only has to write the same file to be evaluated with `src.evaluate.evaluate_track("url", ...)`.

Rules from the proposal that the code enforces: splits grouped by domain (`items.parquet`, column `split`), hyperparameters
chosen on validation only, FPR thresholds set on validation benign scores, test used once per fixed configuration,
bootstrap CIs resample whole domains.

**Run this on a GPU** (Colab, or a local CUDA build of PyTorch). The image VAE trains on about 300k grids per epoch.

In [1]:
# Colab only: clone the repo and move into the project root
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [2]:
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


## 1. Data preparation

Builds `data/processed/items.parquet` (one row per kept item with its split) and `data/processed/qr_grids.npy` (each
code as one bit per module on a 65 x 65 canvas). On Colab, run the download cell of `1_eda.ipynb` first. The image
step reads about 1M PNGs and takes 10 to 30 minutes, once.

In [ ]:
import json, subprocess, sys
from src.config import PROC_DIR

if not (PROC_DIR / "qr_grids.npy").exists():
    subprocess.run([sys.executable, "-m", "src.prepare_data"], check=True)
print(json.dumps(json.loads((PROC_DIR / "prepare_summary.json").read_text()), indent=2))

**Note on the split.** Because the largest domains fill train first, validation and test hold only small, long-tail
domains. Among malicious items, IP-address hosts are about 36% of train but about 2% of validation and test
(`ip_host_share_of_malicious` above). This comes from the rule, not from grouping IPs by their last two octets (grouping by
the full IP gives the same shares). Test therefore measures detection on rarely-seen sites with few IP hosts, and the
report should say so.

## 2. Floors

In [ ]:
import pandas as pd
from src.baselines import run_floors
from src.evaluate import load_score
from src.metrics import auroc
from src.prepare_data import load_items

floors = run_floors()
items = load_items(["item_id", "label", "split"]).set_index("item_id")
rows = {}
for name in floors:
    s = load_score(name)
    v = items.loc[s.index]
    m = (v["split"] == "validation").to_numpy()
    rows[name] = {"validation AUROC": auroc(s[m], v.loc[m, "label"]), "features": ", ".join(floors[name]["features"])}
pd.DataFrame(rows).T

## 3. Image VAE: hyperparameters on validation

Latent size and beta are varied one at a time around a base configuration, on validation data only. Each run reports
the validation benign negative ELBO per module (label-free, also used for early stopping), validation AUROC for both
scores, and the number of active latent units (KL per dimension > 0.01) as the collapse diagnostic.

`SELECT_BY` decides which column picks the configuration. `val_auroc_*` uses validation malicious labels, which the
proposal allows (validation data only). `val_benign_neg_elbo` is the strictly label-free option. State the choice in the report.

In [ ]:
from src.train import run_image_vae, RUNS_DIR

BASE = dict(latent=32, beta=1.0)
LATENTS = [8, 16, 32, 64]
BETAS = [0.25, 0.5, 1.0, 2.0, 4.0]
SWEEP_SEED = 0
EPOCHS = 40

configs = [dict(latent=l, beta=BASE["beta"]) for l in LATENTS] + [dict(latent=BASE["latent"], beta=b) for b in BETAS if b != BASE["beta"]]

def run_info(name):
    return json.loads((RUNS_DIR / f"{name}.json").read_text())

sweep = []
for cfg in configs:
    name = f"image_vae_L{cfg['latent']}_b{cfg['beta']:g}_s{SWEEP_SEED}"
    if not (RUNS_DIR / f"{name}.json").exists():
        run_image_vae(**cfg, seed=SWEEP_SEED, epochs=EPOCHS)
    r = run_info(name)
    last = r["history"][-1]
    sweep.append({**cfg, "run": name, "val_benign_neg_elbo": r["best_val_benign_neg_elbo"],
                  "val_auroc_nll": r["val_auroc"]["nll"], "val_auroc_neg_elbo": r["val_auroc"]["neg_elbo"],
                  "active_units": last["active_units"], "epochs_run": len(r["history"])})
sweep = pd.DataFrame(sweep)
sweep

In [ ]:
SELECT_BY = "val_auroc_neg_elbo"     # or "val_auroc_nll", or "val_benign_neg_elbo" (lower is better)
best = sweep.sort_values(SELECT_BY, ascending=(SELECT_BY == "val_benign_neg_elbo")).iloc[0]
CHOSEN = dict(latent=int(best["latent"]), beta=float(best["beta"]))
SCORE_COL = "nll" if SELECT_BY == "val_auroc_nll" else "neg_elbo"
print("chosen:", CHOSEN, "score:", SCORE_COL)

### KL per epoch (collapse diagnostic)

If KL per item falls to about zero and active units to zero, the decoder ignores the latent (posterior collapse) and the
score is only the decoder's average grid.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for run in sweep["run"]:
    h = pd.DataFrame(run_info(run)["history"])
    axes[0].plot(h["epoch"], h["train_kl_per_item"], label=run.replace("image_vae_", ""))
    axes[1].plot(h["epoch"], h["val_benign_neg_elbo"], label=run.replace("image_vae_", ""))
axes[0].set(xlabel="epoch", ylabel="KL per item (nats)", title="KL per epoch")
axes[1].set(xlabel="epoch", ylabel="validation benign -ELBO per module", title="Validation loss")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4. Chosen configuration, 5 seeds

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
image_runs = []
for s in SEEDS:
    name = f"image_vae_L{CHOSEN['latent']}_b{CHOSEN['beta']:g}_s{s}"
    if not (RUNS_DIR / f"{name}.json").exists():
        run_image_vae(**CHOSEN, seed=s, epochs=EPOCHS)
    image_runs.append(name)
image_runs

## 5. Supervised references

Same encoders as the VAEs (convolutional for the image, character CNN for the URL) with a linear head, trained on
labelled benign and malicious train items, early-stopped on validation AUROC.

In [ ]:
from src.train import run_reference

for kind in ["image", "url"]:
    if not (RUNS_DIR / f"{kind}_ref_s0.json").exists():
        run_reference(kind=kind, seed=0)

## 6. Test evaluation

Run once, after the configuration above is fixed. AUROC, AUPRC (with the prevalence beside it), recall at 1% and 5% FPR
(thresholds from validation benign scores) with domain-bootstrap 95% CIs, then the success criteria.

In [ ]:
from src.evaluate import evaluate_track

pd.set_option("display.max_columns", 40)
image_res = evaluate_track("image", image_runs, SCORE_COL, "image_ref_s0")
print(f"test items (fit the canvas): {image_res['n_test']:,}")
image_res["summary"].round(4)

In [ ]:
crit = image_res["criteria"]
print(json.dumps({k: v for k, v in crit.items() if k != "strata_table"}, indent=2, default=str))
crit["strata_table"].round(4)

### URL track: floors and reference

The URL VAE is evaluated with the same call once its runs exist:
`evaluate_track("url", ["url_vae_..._s0", ..., "url_vae_..._s4"], "neg_elbo", "url_ref_s0")`.

In [ ]:
url_res = evaluate_track("url", [], None, "url_ref_s0")
url_res["summary"].round(4)